# Bước 4 AOI PCB — huấn luyện detector linh kiện trên Kaggle

Notebook tự chứa này thực hiện: tìm/giải nén dataset YOLO, chuẩn hóa data.yaml, audit annotation, trực quan hóa ground truth, fine-tune YOLO26s, đánh giá, xem dự đoán, export best.pt + ONNX và đóng gói báo cáo.

Dataset mặc định: [PCB Component Detection Consolidated Dataset](https://www.kaggle.com/datasets/aryanstein/pcb-component-detection-consolidated-dataset/data), phiên bản 1. Trong Kaggle chọn **Add Input**, tìm đúng slug `aryanstein/pcb-component-detection-consolidated-dataset`, bật GPU và Internet rồi chạy Run All. Preset CONFIG bên dưới đã trỏ tới `components_data_uncropped/data.yaml`. Kaggle Input là read-only; mọi output nằm trong /kaggle/working.

Lưu ý: bước 4 phát hiện lớp hình thái nhìn thấy được. Định danh part number như MCU/PMIC/ADC cần OCR + BOM ở bước 6.4–6.5. Xem README đi kèm để biết định dạng dataset, license và các file cần gửi lại.

In [ ]:
# Cần bật Internet cho cell này trong lần chạy đầu.
%pip install -q "ultralytics==8.4.104" onnx onnxruntime onnxslim

## 1. Cấu hình

- Preset hiện tại dùng Kaggle dataset `aryanstein/pcb-component-detection-consolidated-dataset`, phiên bản 1 (~2,87 GB).
- dataset_source và data_yaml đã trỏ rõ tới thư mục mount và YAML của preset. Chỉ đổi hai khóa này khi dùng dataset khác.
- model_family phải là yolo26; model có thể là tên checkpoint hoặc đường dẫn Kaggle Input. Notebook kiểm tra kiến trúc thật, không suy luận từ tên file.
- Baseline 1280 px phù hợp hơn cho linh kiện nhỏ. Nếu thiếu VRAM, giảm xuống 960 hoặc đổi model thành yolo26n.pt.
- Giữ strict_audit=True. Mặc định file label bị thiếu là lỗi; negative image cần một file .txt rỗng.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import random
import shutil
import sys
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import matplotlib.pyplot as plt
import PIL
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from PIL import Image, ImageDraw

import ultralytics
from ultralytics import YOLO

EXPECTED_ULTRALYTICS_VERSION = "8.4.104"
if ultralytics.__version__ != EXPECTED_ULTRALYTICS_VERSION:
    raise RuntimeError(
        f"Notebook dùng trainer contract đã audit cho Ultralytics {EXPECTED_ULTRALYTICS_VERSION}, "
        f"nhưng runtime là {ultralytics.__version__}. Chạy lại cell cài đặt và restart session."
    )

CONFIG = {
    # Preset Kaggle đã xác minh qua API ngày 2026-08-17.
    "dataset_source": "/kaggle/input/pcb-component-detection-consolidated-dataset",
    "data_yaml": "components_data_uncropped/data.yaml",
    "dataset_name": "PCB Component Detection Consolidated Dataset",
    "dataset_version": "1 (2025-07-06)",
    "dataset_license": "Apache-2.0 declared by Kaggle uploader; verify licenses of upstream component datasets before commercial use",
    "dataset_source_url": "https://www.kaggle.com/datasets/aryanstein/pcb-component-detection-consolidated-dataset/data",
    "model_family": "yolo26",
    "model": "yolo26s.pt",
    "imgsz": 1280,
    "epochs": 100,
    "batch": -1,
    "eval_batch": 4,
    "workers": 2,
    "cache": False,
    "patience": 25,
    "save_period": 10,
    "seed": 42,
    "require_gpu": True,
    "strict_audit": True,
    "decode_max_per_split": 500,
    "hash_duplicates": True,
    # Dataset tile có thể giữ partial object với box vượt mép; Ultralytics chấp nhận xywh chuẩn hóa này.
    "allow_edge_crossing_boxes": True,
    # Loại bản sao ở split ưu tiên thấp hơn bằng image-list trong working, không sửa Kaggle Input.
    "deduplicate_across_splits": True,
    # Public preset có class transducer chỉ ở test; cảnh báo rõ nhưng vẫn cho train các class còn lại.
    "absent_train_class_is_error": False,
    # Safe-by-default: dùng file .txt rỗng cho negative image. Chỉ bật nếu dataset cố ý bỏ file label.
    "allow_negative_images": False,
    "max_missing_label_ratio": 0.0,
    # Giới hạn giải nén thận trọng để chừa dung lượng cho run/model trong Kaggle Working.
    "max_extract_gb": 12,
    "preview_count": 6,
    "predict_count": 8,
    "conf": 0.25,
    "iou": 0.70,
    "max_det": 2000,
    # Khóa cùng một contract cho train/early-stop/val/predict/export: one-to-many + NMS.
    "end2end": False,
    "onnx_opset": 17,
    "onnx_dynamic": False,
    "onnx_simplify": True,
}

MODEL_FAMILY = str(CONFIG["model_family"]).strip().lower()
if MODEL_FAMILY != "yolo26":
    raise ValueError("Notebook này khóa cho YOLO26 detect; CONFIG['model_family'] phải là 'yolo26'.")
if CONFIG["end2end"] is not False:
    raise ValueError("Baseline AOI dùng one-to-many; hãy giữ CONFIG['end2end']=False.")
if not isinstance(CONFIG["imgsz"], int) or CONFIG["imgsz"] <= 0:
    raise ValueError("CONFIG['imgsz'] phải là số nguyên dương.")
missing_ratio_limit = float(CONFIG["max_missing_label_ratio"])
if not 0.0 <= missing_ratio_limit <= 1.0:
    raise ValueError("CONFIG['max_missing_label_ratio'] phải nằm trong [0, 1].")
if not isinstance(CONFIG["allow_negative_images"], bool):
    raise TypeError("CONFIG['allow_negative_images'] phải là bool.")

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
if not INPUT_ROOT.exists():
    # Cho phép dry-run ngoài Kaggle khi cần kiểm tra notebook.
    INPUT_ROOT = Path.cwd() / "input"
    WORK_ROOT = Path.cwd() / "working"

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_NAME = f"pcb_component_{RUN_STAMP}"
DATA_WORK_DIR = WORK_ROOT / "dataset"
OUTPUT_ROOT = WORK_ROOT / "pcb_component_training"
REPORT_DIR = OUTPUT_ROOT / "reports"
RUNS_DIR = OUTPUT_ROOT / "runs"
ARTIFACT_DIR = OUTPUT_ROOT / "artifact"
for directory in (WORK_ROOT, DATA_WORK_DIR, OUTPUT_ROOT, REPORT_DIR, RUNS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
# ARTIFACT_DIR được làm sạch ngay trước export để rerun không trộn file cũ.

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"Python       : {sys.version.split()[0]}")
print(f"Ultralytics  : {ultralytics.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA runtime : {torch.version.cuda}")
print(f"Device       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Run name     : {RUN_NAME}")
print(f"Model family : {MODEL_FAMILY}; end2end={CONFIG['end2end']}; max_det={CONFIG['max_det']}")
if "UNKNOWN" in str(CONFIG["dataset_license"]).upper() or not CONFIG["dataset_source_url"]:
    print("CẢNH BÁO provenance: hãy điền dataset_license và dataset_source_url trước khi bàn giao.")
if CONFIG["require_gpu"] and not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Trong Kaggle: Notebook options > Accelerator > GPU, rồi restart session.")

## 2. Tìm dataset và chuẩn hóa data.yaml

Cell dưới chỉ giải nén ZIP vào working và không ghi vào Kaggle Input. Nếu tìm thấy nhiều dataset, notebook dừng và in danh sách để chọn rõ trong CONFIG.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def safe_extract_zip(zip_path: Path, destination: Path, max_extract_gb: float) -> Path:
    destination.mkdir(parents=True, exist_ok=True)
    limit = int(max_extract_gb * 1024**3)
    with zipfile.ZipFile(zip_path) as archive:
        total = sum(info.file_size for info in archive.infolist())
        if total > limit:
            raise RuntimeError(
                f"ZIP sẽ giải nén {total / 1024**3:.1f} GB, vượt giới hạn CONFIG max_extract_gb={max_extract_gb}."
            )
        root = destination.resolve()
        for info in archive.infolist():
            normalized = info.filename.replace("\\", "/")
            member = PurePosixPath(normalized)
            if member.is_absolute() or ".." in member.parts:
                raise RuntimeError(f"ZIP có đường dẫn không an toàn: {info.filename}")
            target = (destination / Path(*member.parts)).resolve()
            if os.path.commonpath([str(root), str(target)]) != str(root):
                raise RuntimeError(f"ZIP path traversal bị chặn: {info.filename}")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)
    return destination


def load_yaml_if_yolo(path: Path):
    try:
        data = yaml.safe_load(path.read_text(encoding="utf-8"))
    except Exception:
        return None
    has_val = isinstance(data, dict) and ("val" in data or "validation" in data)
    if isinstance(data, dict) and "names" in data and "train" in data and has_val:
        return data
    return None


def find_yolo_yamls(root: Path) -> list[Path]:
    if not root.exists():
        return []
    candidates = sorted(set(root.rglob("*.yaml")) | set(root.rglob("*.yml")))
    return [path for path in candidates if load_yaml_if_yolo(path) is not None]


def resolve_input_config_path(value: str | Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else INPUT_ROOT / path


def find_relative_input_yamls(relative_yaml: Path) -> list[Path]:
    """Tìm YAML theo suffix bên dưới một thư mục mount bất kỳ của Kaggle Input."""
    suffix = tuple(part.lower() for part in relative_yaml.parts)
    matches = []
    if not INPUT_ROOT.exists():
        return matches
    for candidate in INPUT_ROOT.rglob(relative_yaml.name):
        if not candidate.is_file():
            continue
        candidate_parts = tuple(part.lower() for part in candidate.parts)
        if len(candidate_parts) < len(suffix) or candidate_parts[-len(suffix):] != suffix:
            continue
        if load_yaml_if_yolo(candidate) is not None:
            matches.append(candidate.resolve())
    return sorted(set(matches), key=str)


def select_dataset_yaml() -> Path:
    explicit_source = CONFIG["dataset_source"]
    explicit_yaml = CONFIG["data_yaml"]

    if explicit_yaml:
        yaml_path = Path(explicit_yaml)
        if not yaml_path.is_absolute() and explicit_source:
            source = resolve_input_config_path(explicit_source)
            if not source.exists():
                # Kaggle đôi khi đổi tên thư mục mount. Dò YAML theo suffix trước khi dừng.
                matches = find_relative_input_yamls(yaml_path)
                if len(matches) == 1:
                    yaml_path = matches[0]
                elif len(matches) > 1:
                    choices = "\n".join(str(path) for path in matches)
                    raise RuntimeError(
                        f"dataset_source '{source}' không tồn tại và data_yaml khớp nhiều Input:\n{choices}"
                    )
                else:
                    mounted = sorted(str(path) for path in INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
                    mounted_text = "\n".join(mounted) or "(Kaggle Input đang rỗng)"
                    raise FileNotFoundError(
                        f"dataset_source không tồn tại: {source}\n"
                        "Hãy chọn Add Input và gắn dataset "
                        "aryanstein/pcb-component-detection-consolidated-dataset.\n"
                        f"Các Input hiện có:\n{mounted_text}"
                    )
            elif source.suffix.lower() == ".zip":
                if DATA_WORK_DIR.exists():
                    shutil.rmtree(DATA_WORK_DIR)
                extracted_root = safe_extract_zip(source, DATA_WORK_DIR, CONFIG["max_extract_gb"])
                yaml_path = extracted_root / yaml_path
            elif source.is_dir():
                yaml_path = source / yaml_path
            else:
                raise ValueError(f"dataset_source phải là thư mục hoặc ZIP khi data_yaml tương đối: {source}")
        elif not yaml_path.is_absolute():
            # Thử trực tiếp dưới Input; nếu không có, tìm theo suffix bên trong các mount dataset.
            direct_path = INPUT_ROOT / yaml_path
            if direct_path.exists() and load_yaml_if_yolo(direct_path) is not None:
                yaml_path = direct_path
            else:
                matches = find_relative_input_yamls(yaml_path)
                if len(matches) == 1:
                    yaml_path = matches[0]
                elif len(matches) > 1:
                    choices = "\n".join(str(path) for path in matches)
                    raise RuntimeError(
                        f"data_yaml '{yaml_path}' khớp nhiều Kaggle Input. "
                        f"Hãy đặt CONFIG['dataset_source'] rõ ràng. Các YAML tìm thấy:\n{choices}"
                    )
                else:
                    yaml_path = direct_path
        if not yaml_path.exists() or load_yaml_if_yolo(yaml_path) is None:
            raise FileNotFoundError(f"data_yaml không tồn tại hoặc không phải YOLO detect YAML: {yaml_path}")
        return yaml_path.resolve()

    search_root = INPUT_ROOT
    if explicit_source:
        source = resolve_input_config_path(explicit_source)
        if not source.exists():
            raise FileNotFoundError(f"dataset_source không tồn tại: {source}")
        if source.suffix.lower() == ".zip":
            if DATA_WORK_DIR.exists():
                # Chỉ xóa đúng thư mục working riêng của notebook.
                shutil.rmtree(DATA_WORK_DIR)
            search_root = safe_extract_zip(source, DATA_WORK_DIR, CONFIG["max_extract_gb"])
        elif source.suffix.lower() in {".yaml", ".yml"}:
            if load_yaml_if_yolo(source) is None:
                raise ValueError(f"YAML không có train/val/names: {source}")
            return source.resolve()
        elif source.is_dir():
            search_root = source
        else:
            raise ValueError("dataset_source phải là thư mục, ZIP hoặc YAML.")

    yamls = find_yolo_yamls(search_root)
    if not yamls and not explicit_source:
        zips = sorted(INPUT_ROOT.rglob("*.zip")) if INPUT_ROOT.exists() else []
        if len(zips) == 1:
            if DATA_WORK_DIR.exists():
                shutil.rmtree(DATA_WORK_DIR)
            search_root = safe_extract_zip(zips[0], DATA_WORK_DIR, CONFIG["max_extract_gb"])
            yamls = find_yolo_yamls(search_root)
        elif len(zips) > 1:
            choices = "\n".join(str(path) for path in zips)
            raise RuntimeError(f"Có nhiều ZIP. Hãy đặt CONFIG['dataset_source'] thành một trong:\n{choices}")

    if len(yamls) != 1:
        choices = "\n".join(str(path) for path in yamls) or "(không tìm thấy)"
        raise RuntimeError(
            "Notebook cần đúng một YOLO data.yaml. Hãy đặt CONFIG['data_yaml'] hoặc "
            f"CONFIG['dataset_source']. Các YAML tìm thấy:\n{choices}"
        )
    return yamls[0].resolve()


def normalize_names(raw_names) -> list[str]:
    if isinstance(raw_names, list):
        names = [str(name).strip() for name in raw_names]
    elif isinstance(raw_names, dict):
        try:
            mapping = {int(key): str(value).strip() for key, value in raw_names.items()}
        except Exception as exc:
            raise ValueError("Keys trong names phải là class ID nguyên.") from exc
        expected = list(range(len(mapping)))
        if sorted(mapping) != expected:
            raise ValueError(f"Class ID trong names phải liên tục từ 0. Nhận được: {sorted(mapping)}")
        names = [mapping[index] for index in expected]
    else:
        raise ValueError("names trong data.yaml phải là list hoặc dict.")
    if not names or any(not name for name in names):
        raise ValueError("Class map rỗng hoặc có tên class rỗng.")
    if len(set(names)) != len(names):
        raise ValueError("Class map có tên bị lặp; hãy dùng tên duy nhất.")
    return names


def conventional_split_candidates(base: Path, split: str) -> list[Path]:
    aliases = {"train": ["train"], "val": ["val", "valid", "validation"], "test": ["test"]}[split]
    candidates = []
    for alias in aliases:
        candidates.extend([base / "images" / alias, base / alias / "images"])
    return candidates


def resolve_entry(entry: str, dataset_base: Path, yaml_parent: Path, split: str) -> Path:
    raw_entry = str(entry).strip().replace("\\", "/")
    if not raw_entry:
        raise ValueError(f"Split '{split}' có entry rỗng.")
    posix_path = PurePosixPath(raw_entry)
    path = Path(*posix_path.parts)
    leading_parent = not posix_path.is_absolute() and posix_path.parts and posix_path.parts[0] == ".."
    if path.is_absolute():
        candidates = [path]
    elif leading_parent:
        # Kaggle không nên đi ra ngoài dataset đã gắn. Bỏ chuỗi ../ và thử bên trong dataset.
        parts = list(posix_path.parts)
        while parts and parts[0] == "..":
            parts.pop(0)
        stripped = Path(*parts)
        candidates = [yaml_parent / stripped, dataset_base / stripped]
    else:
        candidates = [dataset_base / path, yaml_parent / path]

    tried = []
    for candidate in dict.fromkeys(candidates):
        resolved_candidate = candidate.resolve()
        tried.append(resolved_candidate)
        if resolved_candidate.exists():
            return resolved_candidate

    # Chỉ fallback cấu trúc phổ biến cho export có ../; không che giấu typo tùy ý.
    if leading_parent:
        for base in dict.fromkeys([yaml_parent.resolve(), dataset_base.resolve()]):
            for candidate in conventional_split_candidates(base, split):
                resolved_candidate = candidate.resolve()
                tried.append(resolved_candidate)
                if resolved_candidate.exists():
                    return resolved_candidate
    tried_text = "\n".join(str(candidate) for candidate in dict.fromkeys(tried))
    raise FileNotFoundError(f"Không resolve được split '{split}' từ '{entry}'. Đã thử:\n{tried_text}")


def normalize_dataset_yaml(source_yaml: Path):
    raw = yaml.safe_load(source_yaml.read_text(encoding="utf-8"))
    if "val" not in raw and "validation" in raw:
        raw["val"] = raw["validation"]
    names = normalize_names(raw["names"])
    raw_path = raw.get("path")
    if raw_path:
        path_candidate = Path(str(raw_path))
        dataset_base = path_candidate if path_candidate.is_absolute() else source_yaml.parent / path_candidate
        if not dataset_base.exists():
            raise FileNotFoundError(f"data.yaml khai báo path không tồn tại: {dataset_base}")
    else:
        dataset_base = source_yaml.parent
    dataset_base = dataset_base.resolve()

    resolved = {"path": str(dataset_base), "names": {index: name for index, name in enumerate(names)}}
    for split in ("train", "val", "test"):
        value = raw.get(split)
        if value in (None, ""):
            continue
        values = value if isinstance(value, list) else [value]
        paths = [str(resolve_entry(item, dataset_base, source_yaml.parent, split)) for item in values]
        if len(paths) != len(set(paths)):
            raise ValueError(f"Split '{split}' có nhiều entry resolve tới cùng một path: {paths}")
        resolved[split] = paths if isinstance(value, list) else paths[0]

    resolved_yaml = REPORT_DIR / "data_resolved.yaml"
    resolved_yaml.write_text(yaml.safe_dump(resolved, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return raw, resolved, names, resolved_yaml


SOURCE_YAML = select_dataset_yaml()
RAW_DATA, RESOLVED_DATA, CLASS_NAMES, RESOLVED_YAML = normalize_dataset_yaml(SOURCE_YAML)
print(f"Source YAML   : {SOURCE_YAML}")
print(f"Resolved YAML : {RESOLVED_YAML}")
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")
print(RESOLVED_YAML.read_text(encoding="utf-8"))

## 3. Audit annotation trước khi dùng GPU

Lỗi nghiêm trọng gồm sai số cột, class ID ngoài class map, tâm/kích thước YOLO ngoài miền chuẩn hóa, ảnh hỏng hoặc split không có object. Box partial-object vượt mép nhưng `x_center y_center width height` vẫn thuộc miền Ultralytics được ghi thành cảnh báo tổng hợp. Annotation trùng trong một ảnh được bỏ qua; ảnh trùng giữa các split được loại khỏi split ưu tiên thấp hơn qua image-list trong working. Mặc định mỗi ảnh phải có file label; negative image được biểu diễn bằng file .txt rỗng.

In [ ]:
def expand_split_entries(value) -> list[Path]:
    entries = value if isinstance(value, list) else [value]
    images = []
    for raw_entry in entries:
        entry = Path(str(raw_entry))
        if entry.is_dir():
            images.extend(path for path in entry.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTS)
        elif entry.is_file() and entry.suffix.lower() == ".txt":
            list_errors = []
            for line_number, line in enumerate(entry.read_text(encoding="utf-8-sig").splitlines(), start=1):
                line = line.strip()
                if not line:
                    continue
                candidate = Path(line.replace("\\", "/"))
                if not candidate.is_absolute():
                    candidate = entry.parent / candidate
                candidate = candidate.resolve()
                if not candidate.is_file():
                    list_errors.append(f"line {line_number}: không tồn tại: {candidate}")
                elif candidate.suffix.lower() not in IMAGE_EXTS:
                    list_errors.append(f"line {line_number}: extension không hỗ trợ: {candidate}")
                else:
                    images.append(candidate)
            if list_errors:
                details = "\n".join(list_errors[:20])
                raise FileNotFoundError(f"Split list {entry} có entry lỗi:\n{details}")
        elif entry.is_file() and entry.suffix.lower() in IMAGE_EXTS:
            images.append(entry)
        else:
            raise FileNotFoundError(f"Split entry không đọc được: {entry}")
    return sorted(set(path.resolve() for path in images), key=str)


def label_path_for_image(image_path: Path) -> Path | None:
    parts = list(image_path.parts)
    indices = [index for index, part in enumerate(parts) if part.lower() == "images"]
    if not indices:
        return None
    parts[indices[-1]] = "labels"
    return Path(*parts).with_suffix(".txt")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def add_issue(issues: list[dict], severity: str, code: str, message: str, **context):
    issues.append({"severity": severity, "code": code, "message": message, **{k: str(v) for k, v in context.items()}})


def audit_dataset(resolved: dict, class_names: list[str]):
    split_images = {
        split: expand_split_entries(resolved[split])
        for split in ("train", "val", "test")
        if split in resolved
    }
    issues = []
    invalid_rows = []
    distributions = []
    split_reports = {}
    all_box_areas = []
    all_box_min_sides = []
    edge_crossing_boxes = 0
    max_edge_overshoot = 0.0
    image_hash_locations = defaultdict(list)
    dataset_fingerprint = hashlib.sha256()

    for split, images in split_images.items():
        if not images:
            add_issue(issues, "error", "empty_split", f"Split {split} không có ảnh.", split=split)
            continue

        counts = Counter()
        label_files = 0
        missing_labels = 0
        empty_labels = 0
        missing_label_examples = []
        valid_boxes = 0
        duplicate_rows = 0
        seen_label_targets = {}

        decode_limit = int(CONFIG["decode_max_per_split"])
        decode_sample = images if decode_limit <= 0 or len(images) <= decode_limit else random.Random(CONFIG["seed"]).sample(images, decode_limit)
        for image_path in decode_sample:
            try:
                with Image.open(image_path) as image:
                    image.verify()
            except Exception as exc:
                add_issue(issues, "error", "corrupt_image", str(exc), split=split, image=image_path)

        for image_path in images:
            relative_name = str(image_path)
            dataset_fingerprint.update(f"{split}|{relative_name}|{image_path.stat().st_size}".encode())
            if CONFIG["hash_duplicates"]:
                image_digest = sha256_file(image_path)
                image_hash_locations[image_digest].append((split, str(image_path)))
                dataset_fingerprint.update(image_digest.encode())

            label_path = label_path_for_image(image_path)
            if label_path is None:
                missing_labels += 1
                missing_label_examples.append(str(image_path))
                add_issue(
                    issues,
                    "error",
                    "nonstandard_image_path",
                    "Đường dẫn ảnh không chứa thư mục images nên không suy ra được labels.",
                    split=split,
                    image=image_path,
                )
                continue
            if label_path in seen_label_targets and seen_label_targets[label_path] != image_path:
                add_issue(
                    issues,
                    "error",
                    "shared_label_file",
                    "Hai ảnh đang trỏ tới cùng file label.",
                    split=split,
                    image=image_path,
                    label=label_path,
                )
            seen_label_targets[label_path] = image_path

            if not label_path.exists():
                missing_labels += 1
                missing_label_examples.append(str(label_path))
                continue
            label_files += 1
            text = label_path.read_text(encoding="utf-8-sig").strip()
            dataset_fingerprint.update(text.encode())
            if not text:
                empty_labels += 1
                continue

            row_keys = set()
            for line_number, line in enumerate(text.splitlines(), start=1):
                row = line.strip()
                if not row:
                    continue
                columns = row.split()
                context = {"split": split, "image": image_path, "label": label_path, "line": line_number, "row": row}
                if len(columns) != 5:
                    invalid_rows.append({**context, "reason": f"expected_5_columns_got_{len(columns)}"})
                    add_issue(issues, "error", "invalid_column_count", "YOLO detect cần đúng 5 cột.", **context)
                    continue
                try:
                    values = [float(value) for value in columns]
                except ValueError:
                    invalid_rows.append({**context, "reason": "non_numeric"})
                    add_issue(issues, "error", "non_numeric_label", "Nhãn chứa giá trị không phải số.", **context)
                    continue
                if not all(math.isfinite(value) for value in values):
                    invalid_rows.append({**context, "reason": "non_finite"})
                    add_issue(issues, "error", "non_finite_label", "Nhãn chứa NaN hoặc Inf.", **context)
                    continue
                class_value, x_center, y_center, width, height = values
                if not class_value.is_integer() or not 0 <= int(class_value) < len(class_names):
                    invalid_rows.append({**context, "reason": "invalid_class_id"})
                    add_issue(issues, "error", "invalid_class_id", "Class ID không nguyên hoặc ngoài class map.", **context)
                    continue
                eps = 1e-6
                # Đây là contract mà Ultralytics verify_image_label kiểm tra cho YOLO xywh.
                valid_geometry = (
                    -eps <= x_center <= 1 + eps
                    and -eps <= y_center <= 1 + eps
                    and 0 < width <= 1 + eps
                    and 0 < height <= 1 + eps
                )
                if not valid_geometry:
                    invalid_rows.append({**context, "reason": "invalid_or_out_of_bounds_box"})
                    add_issue(issues, "error", "invalid_box", "Tọa độ YOLO xywh không hợp lệ hoặc ngoài miền chuẩn hóa.", **context)
                    continue
                row_key = tuple(round(value, 8) for value in values)
                if row_key in row_keys:
                    duplicate_rows += 1
                    invalid_rows.append({**context, "reason": "duplicate_annotation"})
                    add_issue(issues, "warning", "duplicate_annotation_ignored", "Annotation trùng được bỏ qua.", **context)
                    continue
                row_keys.add(row_key)
                edge_overshoot = max(
                    width / 2 - x_center,
                    height / 2 - y_center,
                    x_center + width / 2 - 1,
                    y_center + height / 2 - 1,
                    0.0,
                )
                if edge_overshoot > eps:
                    if CONFIG["allow_edge_crossing_boxes"]:
                        edge_crossing_boxes += 1
                        max_edge_overshoot = max(max_edge_overshoot, edge_overshoot)
                    else:
                        invalid_rows.append({**context, "reason": "box_crosses_image_edge"})
                        add_issue(issues, "error", "box_crosses_image_edge", "Box vượt mép ảnh theo policy.", **context)
                        continue
                class_id = int(class_value)
                counts[class_id] += 1
                valid_boxes += 1
                all_box_areas.append(width * height)
                all_box_min_sides.append(min(width, height) * CONFIG["imgsz"])

        for class_id, class_name in enumerate(class_names):
            distributions.append(
                {"split": split, "class_id": class_id, "class_name": class_name, "instances": int(counts[class_id])}
            )
        missing_label_ratio = missing_labels / len(images)
        split_reports[split] = {
            "images": len(images),
            "label_files": label_files,
            "missing_label_files": missing_labels,
            "missing_label_ratio": missing_label_ratio,
            "missing_label_examples": missing_label_examples[:20],
            "empty_label_files": empty_labels,
            "valid_boxes": valid_boxes,
            "duplicate_rows": duplicate_rows,
            "class_counts": {str(key): int(value) for key, value in sorted(counts.items())},
        }
        if missing_labels:
            policy_context = {
                "split": split,
                "missing": missing_labels,
                "images": len(images),
                "ratio": f"{missing_label_ratio:.6f}",
                "limit": CONFIG["max_missing_label_ratio"],
            }
            if not CONFIG["allow_negative_images"]:
                add_issue(
                    issues,
                    "error",
                    "missing_label_files",
                    "Thiếu file label. Tạo file .txt rỗng cho negative image, hoặc bật policy có chủ ý.",
                    **policy_context,
                )
            elif missing_label_ratio > missing_ratio_limit:
                add_issue(
                    issues,
                    "error",
                    "missing_label_ratio_exceeded",
                    "Tỷ lệ file label thiếu vượt policy đã khai báo.",
                    **policy_context,
                )
            else:
                add_issue(
                    issues,
                    "warning",
                    "missing_labels_allowed_by_policy",
                    "File label thiếu được chấp nhận bởi policy tường minh; xác minh đây thật sự là negative image.",
                    **policy_context,
                )
        if valid_boxes == 0:
            add_issue(issues, "error", "no_objects", f"Split {split} không có object hợp lệ.", split=split)
        if split == "train":
            for class_id, class_name in enumerate(class_names):
                if counts[class_id] == 0:
                    severity = "error" if CONFIG["absent_train_class_is_error"] else "warning"
                    add_issue(issues, severity, "class_absent_train", f"Class {class_name} không xuất hiện trong train; model sẽ không học được class này.", class_id=class_id)
        if split in {"val", "test"}:
            for class_id, class_name in enumerate(class_names):
                if counts[class_id] == 0:
                    add_issue(issues, "warning", "class_absent_eval", f"Class {class_name} không có trong {split}; không đo được metric.", class_id=class_id)

    overlaps = []
    excluded_across_splits = set()
    if CONFIG["hash_duplicates"]:
        for digest, locations in image_hash_locations.items():
            if len({split for split, _ in locations}) > 1:
                overlaps.append({"sha256": digest, "locations": locations})
        if overlaps:
            if CONFIG["deduplicate_across_splits"]:
                split_rank = {"train": 0, "val": 1, "test": 2}
                for overlap in overlaps:
                    ordered = sorted(overlap["locations"], key=lambda item: (split_rank[item[0]], item[1]))
                    excluded_across_splits.update(path for _, path in ordered[1:])
                add_issue(
                    issues,
                    "warning",
                    "cross_split_duplicates_excluded",
                    f"Đã loại {len(excluded_across_splits)} ảnh trùng khỏi split ưu tiên thấp hơn.",
                )
            else:
                add_issue(
                    issues,
                    "error",
                    "cross_split_duplicate",
                    f"Có {len(overlaps)} hash ảnh trùng giữa các split.",
                )

    training_split_images = {
        split: [path for path in images if str(path) not in excluded_across_splits]
        for split, images in split_images.items()
    }
    training_resolved = {"path": resolved["path"], "names": resolved["names"]}
    for split, images in training_split_images.items():
        if not images:
            add_issue(issues, "error", "empty_split_after_dedup", f"Split {split} rỗng sau dedup.", split=split)
            continue
        list_path = REPORT_DIR / f"{split}_images.txt"
        list_path.write_text("\n".join(str(path) for path in images) + "\n", encoding="utf-8")
        training_resolved[split] = str(list_path)
        split_reports[split]["training_images_after_cross_split_dedup"] = len(images)
    training_yaml = REPORT_DIR / "data_training.yaml"
    training_yaml.write_text(yaml.safe_dump(training_resolved, sort_keys=False, allow_unicode=True), encoding="utf-8")

    if edge_crossing_boxes:
        add_issue(
            issues,
            "warning",
            "edge_crossing_boxes_allowed",
            "Partial-object boxes vượt mép được giữ vì xywh vẫn hợp lệ với Ultralytics.",
            count=edge_crossing_boxes,
            max_normalized_overshoot=f"{max_edge_overshoot:.8f}",
        )

    area_stats = {}
    if all_box_areas:
        area_array = np.asarray(all_box_areas, dtype=float)
        min_side_array = np.asarray(all_box_min_sides, dtype=float)
        area_stats = {
            "normalized_area_p01": float(np.percentile(area_array, 1)),
            "normalized_area_p10": float(np.percentile(area_array, 10)),
            "normalized_area_p50": float(np.percentile(area_array, 50)),
            "normalized_area_p90": float(np.percentile(area_array, 90)),
            "approx_boxes_min_side_below_4px_at_imgsz": int((min_side_array < 4).sum()),
            "approx_boxes_min_side_below_8px_at_imgsz": int((min_side_array < 8).sum()),
        }
        if area_stats["approx_boxes_min_side_below_8px_at_imgsz"]:
            add_issue(
                issues,
                "warning",
                "tiny_boxes_at_training_size",
                "Một số box có cạnh ước tính dưới 8 px; cân nhắc crop/tiling board trước detection.",
                count=area_stats["approx_boxes_min_side_below_8px_at_imgsz"],
                imgsz=CONFIG["imgsz"],
            )

    report = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_yaml": str(SOURCE_YAML),
        "resolved_yaml": str(RESOLVED_YAML),
        "training_yaml": str(training_yaml),
        "class_names": class_names,
        "dataset_fingerprint_sha256": dataset_fingerprint.hexdigest(),
        "splits": split_reports,
        "box_stats": area_stats,
        "missing_label_policy": {
            "allow_negative_images": CONFIG["allow_negative_images"],
            "max_missing_label_ratio": missing_ratio_limit,
            "explicit_empty_label_files_are_negative": True,
        },
        "cross_split_duplicate_count": len(overlaps),
        "cross_split_duplicate_examples": overlaps[:100],
        "cross_split_excluded_images": sorted(excluded_across_splits),
        "edge_crossing_boxes": {
            "allowed": CONFIG["allow_edge_crossing_boxes"],
            "count": edge_crossing_boxes,
            "max_normalized_overshoot": max_edge_overshoot,
        },
        "issues": issues,
        "error_count": sum(issue["severity"] == "error" for issue in issues),
        "warning_count": sum(issue["severity"] == "warning" for issue in issues),
    }
    invalid_columns = ["split", "image", "label", "line", "row", "reason"]
    invalid_frame = pd.DataFrame(invalid_rows, columns=invalid_columns)
    distribution_columns = ["split", "class_id", "class_name", "instances"]
    distribution_frame = pd.DataFrame(distributions, columns=distribution_columns)
    return report, training_split_images, distribution_frame, invalid_frame, training_resolved, training_yaml


AUDIT, SPLIT_IMAGES, CLASS_DISTRIBUTION, INVALID_LABELS, TRAINING_DATA, TRAINING_YAML = audit_dataset(RESOLVED_DATA, CLASS_NAMES)
(REPORT_DIR / "dataset_audit.json").write_text(json.dumps(AUDIT, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
CLASS_DISTRIBUTION.to_csv(REPORT_DIR / "class_distribution.csv", index=False)
INVALID_LABELS.to_csv(REPORT_DIR / "invalid_labels.csv", index=False)

display(pd.DataFrame(AUDIT["splits"]).T)
display(CLASS_DISTRIBUTION.pivot(index=["class_id", "class_name"], columns="split", values="instances").fillna(0).astype(int))
print(json.dumps(AUDIT["box_stats"], indent=2, ensure_ascii=False))
print(f"Training YAML: {TRAINING_YAML}")
print(f"Audit: {AUDIT['error_count']} error(s), {AUDIT['warning_count']} warning(s)")
if AUDIT["issues"]:
    display(pd.DataFrame(AUDIT["issues"]).head(100))
if CONFIG["strict_audit"] and AUDIT["error_count"]:
    raise RuntimeError(
        f"Dataset audit thất bại với {AUDIT['error_count']} lỗi. "
        f"Xem {REPORT_DIR / 'dataset_audit.json'} và invalid_labels.csv, sửa dữ liệu rồi chạy lại."
    )

## 4. Xem ground-truth ngẫu nhiên

Không train nếu box/tên lớp trong các hình dưới không khớp linh kiện. Audit cú pháp không thể phát hiện mọi lỗi ngữ nghĩa.

In [ ]:
def read_valid_boxes(image_path: Path):
    label_path = label_path_for_image(image_path)
    if label_path is None or not label_path.exists():
        return []
    boxes = []
    for line in label_path.read_text(encoding="utf-8-sig").splitlines():
        columns = line.split()
        if len(columns) != 5:
            continue
        class_id, x_center, y_center, width, height = map(float, columns)
        boxes.append((int(class_id), x_center, y_center, width, height))
    return boxes


def render_ground_truth(image_path: Path) -> Image.Image:
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    image_width, image_height = image.size
    line_width = max(2, round(max(image.size) / 500))
    for class_id, x_center, y_center, width, height in read_valid_boxes(image_path):
        x1 = (x_center - width / 2) * image_width
        y1 = (y_center - height / 2) * image_height
        x2 = (x_center + width / 2) * image_width
        y2 = (y_center + height / 2) * image_height
        color = tuple(int(value) for value in plt.cm.tab20(class_id % 20)[:3] * 255)
        draw.rectangle((x1, y1, x2, y2), outline=color, width=line_width)
        draw.text((x1 + 2, max(0, y1 - 12)), CLASS_NAMES[class_id], fill=color)
    return image


preview_pool = SPLIT_IMAGES.get("train", []) + SPLIT_IMAGES.get("val", [])
preview_count = min(CONFIG["preview_count"], len(preview_pool))
preview_paths = random.Random(CONFIG["seed"]).sample(preview_pool, preview_count)
columns = 2
rows = max(1, math.ceil(preview_count / columns))
fig, axes = plt.subplots(rows, columns, figsize=(16, 7 * rows))
axes = np.atleast_1d(axes).ravel()
for axis, image_path in zip(axes, preview_paths):
    axis.imshow(render_ground_truth(image_path))
    axis.set_title(image_path.name)
    axis.axis("off")
for axis in axes[len(preview_paths):]:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 5. Fine-tune detector

Baseline dùng checkpoint COCO pretrained. Phép xoay/lật phù hợp bước phát hiện hình thái; polarity/orientation được kiểm tra riêng ở 6.3. YOLO26 được ép one-to-many ngay khi trainer tạo model, trước loss/EMA/validator, để checkpoint selection và deployment dùng cùng một contract.

In [ ]:
from ultralytics.models.yolo.detect import DetectionTrainer


def yolo26_contract_state(model_or_wrapper, stage: str) -> dict:
    core = model_or_wrapper.model if isinstance(model_or_wrapper, YOLO) else model_or_wrapper
    if not isinstance(core, torch.nn.Module) or not hasattr(core, "model") or not len(core.model):
        raise RuntimeError(f"{stage}: checkpoint không phải Ultralytics detection model hợp lệ.")
    head = core.model[-1]
    if not hasattr(head, "one2one") or not hasattr(head, "one2many"):
        raise RuntimeError(
            f"{stage}: CONFIG model_family='yolo26' nhưng head {type(head).__name__} "
            "không có đủ one2one/one2many. Kiểm tra checkpoint thay vì tên file."
        )
    return {
        "head_class": type(head).__name__,
        "end2end": bool(getattr(core, "end2end", False)),
        "max_det": int(getattr(head, "max_det", -1)),
    }


def configure_yolo26_contract(core: torch.nn.Module, stage: str):
    yolo26_contract_state(core, stage)  # architecture gate before mutation
    core.end2end = CONFIG["end2end"]
    core.set_head_attr(max_det=int(CONFIG["max_det"]))
    core.criterion = None  # loss phải được khởi tạo lại theo head mode đã khóa
    state = yolo26_contract_state(core, stage)
    expected = {"end2end": CONFIG["end2end"], "max_det": int(CONFIG["max_det"])}
    if any(state[key] != value for key, value in expected.items()):
        raise RuntimeError(f"{stage}: không áp dụng được model contract. state={state}, expected={expected}")
    return core


class AOIYOLO26DetectionTrainer(DetectionTrainer):
    """Pin v8.4.104: public end2end arg không đổi head khi DetectionTrainer rebuild model."""

    def get_model(self, cfg: str | None = None, weights: str | None = None, verbose: bool = True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)
        return configure_yolo26_contract(model, "trainer.get_model")


train_model = YOLO(CONFIG["model"])
if train_model.task != "detect":
    raise RuntimeError(f"Checkpoint task phải là detect, nhận được: {train_model.task}")
yolo26_contract_state(train_model, "checkpoint ban đầu")
train_result = train_model.train(
    trainer=AOIYOLO26DetectionTrainer,
    data=str(TRAINING_YAML),
    epochs=CONFIG["epochs"],
    imgsz=CONFIG["imgsz"],
    batch=CONFIG["batch"],
    workers=CONFIG["workers"],
    cache=CONFIG["cache"],
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=False,
    pretrained=True,
    optimizer="auto",
    patience=CONFIG["patience"],
    save_period=CONFIG["save_period"],
    end2end=CONFIG["end2end"],
    max_det=CONFIG["max_det"],
    seed=CONFIG["seed"],
    deterministic=True,
    amp=True,
    plots=True,
    verbose=True,
    degrees=180.0,
    translate=0.05,
    scale=0.25,
    shear=2.0,
    perspective=0.0005,
    flipud=0.5,
    fliplr=0.5,
    hsv_h=0.01,
    hsv_s=0.30,
    hsv_v=0.20,
    mosaic=0.50,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=10,
)

if not isinstance(train_model.trainer, AOIYOLO26DetectionTrainer):
    raise RuntimeError("Ultralytics không dùng AOIYOLO26DetectionTrainer như yêu cầu.")
if bool(train_model.trainer.args.end2end) != CONFIG["end2end"]:
    raise RuntimeError("Trainer args.end2end không khớp CONFIG.")
if int(train_model.trainer.args.max_det) != int(CONFIG["max_det"]):
    raise RuntimeError("Trainer args.max_det không khớp CONFIG.")
TRAIN_MODEL_CONTRACT = yolo26_contract_state(train_model.trainer.model, "model sau train")
if TRAIN_MODEL_CONTRACT["end2end"] != CONFIG["end2end"] or TRAIN_MODEL_CONTRACT["max_det"] != int(CONFIG["max_det"]):
    raise RuntimeError(f"Model sau train sai contract: {TRAIN_MODEL_CONTRACT}")
TRAIN_VALIDATOR_CONTRACT = {
    "args_end2end": bool(train_model.trainer.validator.args.end2end),
    "runtime_end2end": bool(getattr(train_model.trainer.validator, "end2end", True)),
    "max_det": int(train_model.trainer.validator.args.max_det),
}
if TRAIN_VALIDATOR_CONTRACT != {
    "args_end2end": CONFIG["end2end"],
    "runtime_end2end": CONFIG["end2end"],
    "max_det": int(CONFIG["max_det"]),
}:
    raise RuntimeError(f"Internal validation sai contract: {TRAIN_VALIDATOR_CONTRACT}")

TRAIN_SAVE_DIR = Path(train_model.trainer.save_dir)
BEST_PT_SOURCE = Path(train_model.trainer.best)
if not BEST_PT_SOURCE.exists():
    fallback = TRAIN_SAVE_DIR / "weights" / "best.pt"
    if not fallback.exists():
        raise FileNotFoundError("Train kết thúc nhưng không tìm thấy best.pt.")
    BEST_PT_SOURCE = fallback
raw_best_fitness = train_model.trainer.best_fitness
best_fitness_value = float(raw_best_fitness) if raw_best_fitness is not None else None
if best_fitness_value is not None and not math.isfinite(best_fitness_value):
    best_fitness_value = None
TRAINING_SUMMARY = {
    "epochs_completed": int(getattr(train_model.trainer, "epoch", -1)) + 1,
    "best_epoch": int(getattr(getattr(train_model.trainer, "stopper", None), "best_epoch", -1)),
    "best_fitness": best_fitness_value,
    "model_contract": TRAIN_MODEL_CONTRACT,
    "internal_validator_contract": TRAIN_VALIDATOR_CONTRACT,
}
print(f"Train output : {TRAIN_SAVE_DIR}")
print(f"Best model   : {BEST_PT_SOURCE}")
print(f"Train contract: {TRAIN_MODEL_CONTRACT}")

## 6. Đánh giá best.pt

Validation dùng one-to-many head + NMS cho board dày linh kiện và max_det=2000; mức này tránh truncate board AOI có thể vượt 1000 linh kiện. Nếu dataset có test split, notebook đánh giá thêm test sau val.

In [ ]:
best_model = YOLO(str(BEST_PT_SOURCE))
if best_model.task != "detect":
    raise RuntimeError(f"best.pt task phải là detect, nhận được: {best_model.task}")
BEST_PT_CONTRACT = yolo26_contract_state(best_model, "best.pt")
if BEST_PT_CONTRACT["end2end"] != CONFIG["end2end"] or BEST_PT_CONTRACT["max_det"] != int(CONFIG["max_det"]):
    raise RuntimeError(f"best.pt không giữ đúng train contract: {BEST_PT_CONTRACT}")
model_names = best_model.names
if isinstance(model_names, dict):
    trained_class_names = [str(model_names[index]) for index in range(len(model_names))]
else:
    trained_class_names = [str(name) for name in model_names]
if trained_class_names != CLASS_NAMES:
    raise RuntimeError(
        f"Class map trong best.pt không khớp data.yaml. Model={trained_class_names}, data={CLASS_NAMES}"
    )


def evaluate_split(model: YOLO, split: str):
    kwargs = dict(
        data=str(TRAINING_YAML),
        split=split,
        imgsz=CONFIG["imgsz"],
        batch=CONFIG["eval_batch"],
        device=DEVICE,
        conf=0.001,
        iou=CONFIG["iou"],
        max_det=CONFIG["max_det"],
        plots=True,
        project=str(RUNS_DIR),
        name=f"{RUN_NAME}_{split}",
        exist_ok=False,
        verbose=True,
    )
    kwargs["end2end"] = CONFIG["end2end"]
    metrics = model.val(**kwargs)
    save_dir = getattr(metrics, "save_dir", None)
    if save_dir is None:
        raise RuntimeError(f"Ultralytics không trả save_dir cho split {split}.")
    return metrics, Path(save_dir)


VAL_METRICS, VAL_SAVE_DIR = evaluate_split(best_model, "val")
if "test" in RESOLVED_DATA:
    TEST_METRICS, TEST_SAVE_DIR = evaluate_split(best_model, "test")
else:
    TEST_METRICS, TEST_SAVE_DIR = None, None


def scalar(value):
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return None
    return numeric if math.isfinite(numeric) else None


def metrics_summary(metrics):
    return {
        "precision_macro": scalar(metrics.box.mp),
        "recall_macro": scalar(metrics.box.mr),
        "map50": scalar(metrics.box.map50),
        "map75": scalar(metrics.box.map75),
        "map50_95": scalar(metrics.box.map),
        "fitness": scalar(getattr(metrics, "fitness", None)),
        "speed_ms_per_image": {key: scalar(value) for key, value in getattr(metrics, "speed", {}).items()},
    }


def per_class_table(metrics, split: str) -> pd.DataFrame:
    maps = np.asarray(metrics.box.maps, dtype=float)
    precision = np.asarray(getattr(metrics.box, "p", []), dtype=float)
    recall = np.asarray(getattr(metrics.box, "r", []), dtype=float)
    f1 = np.asarray(getattr(metrics.box, "f1", []), dtype=float)
    evaluated_ids = np.asarray(
        getattr(metrics.box, "ap_class_index", np.arange(len(precision))), dtype=int
    )
    metric_position = {int(class_id): position for position, class_id in enumerate(evaluated_ids)}
    rows = []
    for class_id, class_name in enumerate(CLASS_NAMES):
        position = metric_position.get(class_id)
        rows.append(
            {
                "split": split,
                "class_id": class_id,
                "class_name": class_name,
                "precision": float(precision[position]) if position is not None and position < len(precision) else np.nan,
                "recall": float(recall[position]) if position is not None and position < len(recall) else np.nan,
                "f1": float(f1[position]) if position is not None and position < len(f1) else np.nan,
                # metrics.box.maps điền macro mAP cho class vắng; tránh báo cáo nhầm bằng NaN.
                "map50_95": float(maps[class_id]) if position is not None and class_id < len(maps) else np.nan,
            }
        )
    return pd.DataFrame(rows)


METRICS_SUMMARY = {"val": metrics_summary(VAL_METRICS)}
PER_CLASS = per_class_table(VAL_METRICS, "val")
if TEST_METRICS is not None:
    METRICS_SUMMARY["test"] = metrics_summary(TEST_METRICS)
    PER_CLASS = pd.concat([PER_CLASS, per_class_table(TEST_METRICS, "test")], ignore_index=True)

(REPORT_DIR / "metrics_summary.json").write_text(
    json.dumps(METRICS_SUMMARY, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8"
)
PER_CLASS.to_csv(REPORT_DIR / "per_class_metrics.csv", index=False)
display(pd.DataFrame(METRICS_SUMMARY).T)
display(PER_CLASS)

## 7. Trực quan hóa dự đoán validation

Hãy kiểm tra false negative ở linh kiện nhỏ, box dính nhiều linh kiện và nhầm class có package giống nhau.

In [ ]:
val_pool = SPLIT_IMAGES["val"]
prediction_count = min(CONFIG["predict_count"], len(val_pool))
prediction_paths = random.Random(CONFIG["seed"] + 1).sample(val_pool, prediction_count)
predict_kwargs = dict(
    source=[str(path) for path in prediction_paths],
    imgsz=CONFIG["imgsz"],
    conf=CONFIG["conf"],
    iou=CONFIG["iou"],
    max_det=CONFIG["max_det"],
    device=DEVICE,
    save=True,
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}_predictions",
    exist_ok=False,
    verbose=False,
)
predict_kwargs["end2end"] = CONFIG["end2end"]
PREDICTIONS = best_model.predict(**predict_kwargs)
PREDICTION_SAVE_DIR = Path(best_model.predictor.save_dir)

columns = 2
rows = max(1, math.ceil(len(PREDICTIONS) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(16, 7 * rows))
axes = np.atleast_1d(axes).ravel()
for axis, result in zip(axes, PREDICTIONS):
    axis.imshow(result.plot()[:, :, ::-1])
    axis.set_title(Path(result.path).name)
    axis.axis("off")
for axis in axes[len(PREDICTIONS):]:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 8. Export ONNX và smoke-test

ONNX được export từ best.pt. Baseline để dynamic=False nhằm giảm khác biệt runtime; app vẫn letterbox ảnh về imgsz trước inference. Với YOLO26, export one-to-many không gắn NMS vào graph; Ultralytics wrapper thực hiện NMS khi đọc ONNX.

In [ ]:
import onnx
import onnxruntime as ort

artifact_target = ARTIFACT_DIR.resolve()
if artifact_target.parent != OUTPUT_ROOT.resolve():
    raise RuntimeError(f"Từ chối xóa artifact path ngoài OUTPUT_ROOT: {artifact_target}")
if artifact_target.exists():
    shutil.rmtree(artifact_target)
artifact_target.mkdir(parents=True, exist_ok=False)

BEST_PT = ARTIFACT_DIR / "best.pt"
shutil.copy2(BEST_PT_SOURCE, BEST_PT)

export_kwargs = dict(
    format="onnx",
    imgsz=CONFIG["imgsz"],
    opset=CONFIG["onnx_opset"],
    dynamic=CONFIG["onnx_dynamic"],
    simplify=CONFIG["onnx_simplify"],
    batch=1,
    device="cpu",
    nms=False,
)
export_kwargs["end2end"] = CONFIG["end2end"]

export_model = YOLO(str(BEST_PT))
export_contract = yolo26_contract_state(export_model, "best.pt trước export")
if export_contract != BEST_PT_CONTRACT:
    raise RuntimeError(f"Checkpoint copy làm thay đổi contract: {export_contract} != {BEST_PT_CONTRACT}")
try:
    exported_path = Path(export_model.export(**export_kwargs))
except Exception as first_error:
    if not export_kwargs["simplify"]:
        raise
    print(f"Export simplify=True thất bại: {first_error}\nThử lại simplify=False...")
    export_kwargs["simplify"] = False
    exported_path = Path(export_model.export(**export_kwargs))

BEST_ONNX = ARTIFACT_DIR / "best.onnx"
if exported_path.resolve() != BEST_ONNX.resolve():
    shutil.copy2(exported_path, BEST_ONNX)
onnx_model = onnx.load(str(BEST_ONNX))
onnx.checker.check_model(onnx_model)
ort_session = ort.InferenceSession(str(BEST_ONNX), providers=["CPUExecutionProvider"])
onnx_inputs = [{"name": item.name, "shape": item.shape, "type": item.type} for item in ort_session.get_inputs()]
onnx_outputs = [{"name": item.name, "shape": item.shape, "type": item.type} for item in ort_session.get_outputs()]
expected_input_shape = [1, 3, CONFIG["imgsz"], CONFIG["imgsz"]]
if len(onnx_inputs) != 1 or list(onnx_inputs[0]["shape"]) != expected_input_shape:
    raise RuntimeError(f"ONNX input contract sai: {onnx_inputs}; expected={expected_input_shape}")

# Smoke-test qua Ultralytics ONNX backend trên một ảnh validation.
onnx_wrapper = YOLO(str(BEST_ONNX), task="detect")
onnx_smoke = onnx_wrapper.predict(
    source=str(SPLIT_IMAGES["val"][0]),
    imgsz=CONFIG["imgsz"],
    conf=CONFIG["conf"],
    iou=CONFIG["iou"],
    max_det=CONFIG["max_det"],
    device="cpu",
    end2end=CONFIG["end2end"],
    verbose=False,
)
ONNX_VERIFICATION = {
    "checker_passed": True,
    "ultralytics_smoke_passed": True,
    "smoke_image": str(SPLIT_IMAGES["val"][0]),
    "smoke_detection_count": len(onnx_smoke[0].boxes),
    "inputs": onnx_inputs,
    "outputs": onnx_outputs,
}
(REPORT_DIR / "onnx_verification.json").write_text(
    json.dumps(ONNX_VERIFICATION, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8"
)
print(f"ONNX hợp lệ : {BEST_ONNX}")
print(f"ONNX size    : {BEST_ONNX.stat().st_size / 1024**2:.1f} MB")
print(f"Smoke boxes  : {len(onnx_smoke[0].boxes)}")

## 9. Tạo manifest và gói bàn giao

Cell cuối chỉ lấy best checkpoint, ONNX, metric/audit và plot cần thiết; không đưa last.pt vào ZIP để tránh tăng gấp đôi dung lượng.

In [ ]:
def copy_if_exists(source: Path, destination_name: str | None = None):
    if source.exists() and source.is_file():
        destination = ARTIFACT_DIR / (destination_name or source.name)
        if source.resolve() != destination.resolve():
            shutil.copy2(source, destination)
        return destination
    return None


# Báo cáo do notebook tạo.
for report_name in (
    "data_resolved.yaml",
    "data_training.yaml",
    "train_images.txt",
    "val_images.txt",
    "test_images.txt",
    "dataset_audit.json",
    "class_distribution.csv",
    "invalid_labels.csv",
    "metrics_summary.json",
    "per_class_metrics.csv",
    "onnx_verification.json",
):
    copy_if_exists(REPORT_DIR / report_name)

# File train quan trọng và plot từ train/val.
copy_if_exists(TRAIN_SAVE_DIR / "results.csv")
copy_if_exists(TRAIN_SAVE_DIR / "args.yaml")
plot_names = {
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "F1_curve.png",
    "labels.jpg",
    "labels_correlogram.jpg",
}
for path in TRAIN_SAVE_DIR.rglob("*"):
    if path.is_file() and path.name in plot_names:
        copy_if_exists(path, f"train_{path.name}")
for prefix, eval_dir in (("val", VAL_SAVE_DIR), ("test", TEST_SAVE_DIR)):
    if eval_dir is None:
        continue
    for path in Path(eval_dir).rglob("*"):
        if path.is_file() and path.name in plot_names:
            copy_if_exists(path, f"{prefix}_{path.name}")

# Một số ảnh prediction để review nhanh khi nhận gói bàn giao.
prediction_dir = PREDICTION_SAVE_DIR
if prediction_dir.exists():
    prediction_files = [
        path for path in sorted(prediction_dir.rglob("*"), key=str)
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS
    ]
    for index, path in enumerate(prediction_files[: CONFIG["predict_count"]]):
        copy_if_exists(path, f"prediction_{index:02d}_{path.name}")

runtime = {
    "python": sys.version,
    "platform": platform.platform(),
    "ultralytics": ultralytics.__version__,
    "onnx": onnx.__version__,
    "onnxruntime": ort.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": PIL.__version__,
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(ARTIFACT_DIR / "runtime.json").write_text(json.dumps(runtime, indent=2, allow_nan=False), encoding="utf-8")

handoff_text = f"""AOI PCB component detector — Kaggle handoff
Created UTC: {datetime.now(timezone.utc).isoformat()}
Run: {RUN_NAME}
Classes: {CLASS_NAMES}
Input size: {CONFIG['imgsz']}
Head contract: one-to-many, external NMS, max_det={CONFIG['max_det']}

Primary files:
- best.onnx: preferred inference artifact; one-to-many YOLO output without NMS embedded. Use the Ultralytics wrapper or apply NMS in the runtime.
- best.pt: trusted-environment fallback for debugging or Ultralytics Python integration.
- model_manifest.json: authoritative class order, preprocessing/export flags, metrics and hashes.
- dataset_audit.json: data-quality evidence.

Do not reorder class names in the app. Validate on images from the real AOI camera before production use.
"""
(ARTIFACT_DIR / "HANDOFF.txt").write_text(handoff_text, encoding="utf-8")

manifest = {
    "schema_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_name": RUN_NAME,
    "task": "detect",
    "ultralytics_license_notice": "AGPL-3.0 or Enterprise; verify the applicable license before commercial deployment",
    "pipeline_step": "4_component_detection",
    "model_family": MODEL_FAMILY,
    "base_model": CONFIG["model"],
    "class_names": CLASS_NAMES,
    "class_map": {str(index): name for index, name in enumerate(CLASS_NAMES)},
    "input": {
        "imgsz": CONFIG["imgsz"],
        "channels": 3,
        "color_order_for_ultralytics_api": "BGR numpy or image path; wrapper handles preprocessing",
        "letterbox": True,
    },
    "inference": {
        "recommended_conf": CONFIG["conf"],
        "iou_nms": CONFIG["iou"],
        "head": "one-to-many",
        "max_det": CONFIG["max_det"],
        "end2end": CONFIG["end2end"],
    },
    "onnx": {
        "opset": CONFIG["onnx_opset"],
        "dynamic": CONFIG["onnx_dynamic"],
        "simplified": export_kwargs["simplify"],
        "nms_embedded": False,
        "batch": 1,
        "fixed_input_shape": expected_input_shape,
        "output_contract": "raw one-to-many predictions; apply NMS externally or use Ultralytics wrapper",
        "verification": ONNX_VERIFICATION,
    },
    "training_summary": TRAINING_SUMMARY,
    "train_config": CONFIG,
    "metrics": METRICS_SUMMARY,
    "dataset": {
        "fingerprint_sha256": AUDIT["dataset_fingerprint_sha256"],
        "name": CONFIG["dataset_name"],
        "version": CONFIG["dataset_version"],
        "license": CONFIG["dataset_license"],
        "source_url": CONFIG["dataset_source_url"],
        "source_yaml": str(SOURCE_YAML),
        "training_yaml": str(TRAINING_YAML),
        "splits": AUDIT["splits"],
        "cross_split_excluded_images": AUDIT["cross_split_excluded_images"],
        "edge_crossing_boxes": AUDIT["edge_crossing_boxes"],
        "audit_error_count": AUDIT["error_count"],
        "audit_warning_count": AUDIT["warning_count"],
    },
    "runtime": runtime,
    "files": {},
}
required_before_manifest = {
    "best.pt",
    "best.onnx",
    "dataset_audit.json",
    "metrics_summary.json",
    "per_class_metrics.csv",
    "class_distribution.csv",
    "data_resolved.yaml",
    "data_training.yaml",
    "train_images.txt",
    "val_images.txt",
    "onnx_verification.json",
    "runtime.json",
    "HANDOFF.txt",
}
missing_artifacts = sorted(name for name in required_before_manifest if not (ARTIFACT_DIR / name).is_file())
if missing_artifacts:
    raise FileNotFoundError(f"Không đóng gói vì thiếu artifact bắt buộc: {missing_artifacts}")
for path in sorted(ARTIFACT_DIR.iterdir(), key=lambda item: item.name):
    if path.is_file() and path.name != "model_manifest.json":
        manifest["files"][path.name] = {
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
(ARTIFACT_DIR / "model_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False, default=str, allow_nan=False), encoding="utf-8"
)
if not (ARTIFACT_DIR / "model_manifest.json").is_file():
    raise FileNotFoundError("Không tạo được model_manifest.json.")

BUNDLE_BASE = WORK_ROOT / "pcb_component_detector_artifacts"
BUNDLE_PATH = Path(shutil.make_archive(str(BUNDLE_BASE), "zip", root_dir=ARTIFACT_DIR))
print(f"\nHOÀN TẤT — tải file này từ Kaggle Output:\n{BUNDLE_PATH}")
print(f"Dung lượng: {BUNDLE_PATH.stat().st_size / 1024**2:.1f} MB")
print("Nếu không gửi được cả ZIP, ưu tiên best.onnx + model_manifest.json + metrics_summary.json + dataset_audit.json; best.pt là fallback tin cậy.")

## Hoàn tất

Tải /kaggle/working/pcb_component_detector_artifacts.zip ở tab Output rồi gửi lại. Kèm 10–20 ảnh PCB nguyên bản chưa dùng để train, nguồn/license dataset và mô tả cách chia board/SKU/lot. Không cần gửi toàn bộ dataset nếu không có quyền chia sẻ.